# Suffolk ResStock Heat-Pump Scenario Analysis

Reconstructed notebook for the CELT / Massachusetts Heat Pump Adoption project.

This notebook documents Astika Tyagi's Suffolk County ResStock scenario analysis. It does **not** reproduce collaborator-authored NSI matching code.

Main outputs:
- baseline vs heat-pump upgrade
- annual / winter / heating electricity
- gas and oil reductions
- peak-demand changes
- 10%, 20%, 50%, and 100% adoption scenarios


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)


## Paths

Known saved outputs from the SCC analysis are listed below. Update the paths if the project folders have moved.


In [ ]:
PHASE1_DIR = Path("../data/raw")
WEIGHTED_SUMMARY_PATH = PHASE1_DIR / "phase1_weighted_summary.csv"
HOURLY_PROFILE_PATH = PHASE1_DIR / "phase1_hourly_load_profile.csv"
BUILDING_SUMMARY_PATH = PHASE1_DIR / "up00_building_summary.csv"

for p in [WEIGHTED_SUMMARY_PATH, HOURLY_PROFILE_PATH, BUILDING_SUMMARY_PATH]:
    print(p, p.exists())


## Reconstructed full-applicable scenario

Recovered results from the completed analysis:

- Annual electricity: 2444.93 → 2895.84 GWh (+18.44%)
- Earlier Phase 1 winter metric: 1215.96 → 1595.47 GWh (+31.21%)
- Electric heating: 552.36 → 1029.95 GWh (+86.47%)
- Natural gas: -33.80%
- Fuel oil: -48.42%
- Cooling electricity: +4.7%
- Hourly peak: 741.88 → 1439.52 MW (+94.04%)
- 15-minute peak: 760.96 → 1484.92 MW (+95.14%)

Note: the earlier Phase 1 winter metric is not identical to the later explicit Oct-Apr monthly validation metric.


In [ ]:
full_summary = pd.DataFrame({
    "metric": [
        "Annual electricity (GWh)",
        "Winter electricity - earlier Phase 1 metric (GWh)",
        "Electric heating (GWh)",
        "Hourly peak (MW)",
        "15-min peak (MW)"
    ],
    "baseline": [2444.93, 1215.96, 552.36, 741.88, 760.96],
    "full_applicable_hp": [2895.84, 1595.47, 1029.95, 1439.52, 1484.92],
})
full_summary["change_pct"] = (
    (full_summary["full_applicable_hp"] - full_summary["baseline"])
    / full_summary["baseline"] * 100
)
full_summary


## Partial adoption scenarios

The original analysis evaluated partial adoption among 458 eligible buildings using 500 random draws and seed 42.


In [ ]:
scenario_results = pd.DataFrame({
    "adoption_pct": [0, 10, 20, 50, 100],
    "annual_electricity_change_pct": [0, 1.83, 3.72, 9.20, 18.44],
    "winter_electricity_change_pct": [0, 3.11, 6.30, 15.57, 31.21],
    "heating_electricity_change_pct": [0, 8.62, 17.46, 43.16, 86.47],
    "natural_gas_change_pct": [0, -3.38, -6.85, -16.91, -33.80],
    "fuel_oil_change_pct": [0, -4.89, -9.80, -24.16, -48.42],
})
scenario_results


In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
for c, label in [
    ("annual_electricity_change_pct","Annual electricity"),
    ("winter_electricity_change_pct","Winter electricity"),
    ("heating_electricity_change_pct","Heating electricity"),
]:
    ax.plot(scenario_results["adoption_pct"], scenario_results[c], marker="o", label=label)
ax.axhline(0, linewidth=1)
ax.set_xlabel("Heat-pump adoption among eligible buildings (%)")
ax.set_ylabel("Change from baseline (%)")
ax.set_title("Suffolk ResStock: Electricity Response to Heat-Pump Adoption")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
for c, label in [
    ("natural_gas_change_pct","Natural gas"),
    ("fuel_oil_change_pct","Fuel oil"),
]:
    ax.plot(scenario_results["adoption_pct"], scenario_results[c], marker="o", label=label)
ax.axhline(0, linewidth=1)
ax.set_xlabel("Heat-pump adoption among eligible buildings (%)")
ax.set_ylabel("Change from baseline (%)")
ax.set_title("Suffolk ResStock: Fossil-Fuel Response to Heat-Pump Adoption")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


## Optional: inspect original saved results

This cell runs only when the SCC file exists.


In [ ]:
if WEIGHTED_SUMMARY_PATH.exists():
    weighted_summary = pd.read_csv(WEIGHTED_SUMMARY_PATH)
    display(weighted_summary.head())
    print(weighted_summary.columns.tolist())
else:
    print("Update WEIGHTED_SUMMARY_PATH to rerun this section.")


## Interpretation

The scenarios show that electrification reduces gas and oil use but increases electricity demand, especially heating-season demand and peak load.

Because the later observed-data validation found a winter-heavy baseline mismatch, absolute GWh/MW values should be described as model outputs unless calibrated. Relative scenario changes are the more defensible headline result.
